# Laboratorio 3 — Exploración de datos (EDA) para redes convolucionales

## Contexto
Este notebook realiza una **exploración concisa** del dataset elegido, con el objetivo de entender su estructura antes de diseñar una arquitectura convolucional.

---

## Elección del dataset: **Fashion-MNIST**

**Justificación para el uso de capas convolucionales:**

- **Estructura espacial**: Las imágenes son tensores 2D (28×28) donde la posición de píxeles tiene significado (formas, bordes, texturas). Las convoluciones explotan esta estructura mediante filtros locales.
- **Invariancia traslacional**: Las clases (prendas, calzado) son reconocibles aunque la figura esté desplazada; los kernels compartidos y el pooling ayudan a capturar patrones independientes de la posición.
- **Tamaño manejable**: 70 000 imágenes en escala de grises caben en memoria en un portátil, lo que permite iterar rápido en diseño de arquitectura y experimentos.
- **Múltiples clases**: 10 clases balanceadas permiten evaluar capacidad de discriminación y generalización de la arquitectura.
- **Benchmark establecido**: Sustituto directo de MNIST con mayor dificultad, ampliamente usado para comparar diseños de CNN.

---
## 1. Carga del dataset y dependencias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Carga mediante Keras (TensorFlow)
from tensorflow.keras.datasets import fashion_mnist

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Nombres de las 10 clases (Fashion-MNIST)
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
print("Dataset cargado correctamente.")

---
## 2. Tamaño del dataset y dimensiones de las imágenes

In [ ]:
n_train, n_test = X_train.shape[0], X_test.shape[0]
n_total = n_train + n_test
h, w = X_train.shape[1], X_train.shape[2]

print("Tamaño del dataset:")
print(f"  Entrenamiento: {n_train:,} muestras")
print(f"  Test:          {n_test:,} muestras")
print(f"  Total:         {n_total:,} muestras")
print()
print("Dimensiones de cada imagen:")
print(f"  Altura: {h} px, Ancho: {w} px")
print(f"  Canales: 1 (escala de grises)")
print(f"  Forma típica (N, H, W) = (batch, {h}, {w})")
print()
print("Memoria aproximada (train, uint8):", f"{X_train.nbytes / 1024**2:.2f} MB")

---
## 3. Distribución de clases

In [ ]:
# Conteo por clase (entrenamiento)
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts = pd.Series(y_test).value_counts().sort_index()

df_dist = pd.DataFrame({
    'Clase': [CLASS_NAMES[i] for i in train_counts.index],
    'Train': train_counts.values,
    'Test': test_counts.values
})
df_dist['Total'] = df_dist['Train'] + df_dist['Test']
print("Distribución por clase:")
display(df_dist)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras: entrenamiento
axes[0].bar(df_dist['Clase'], df_dist['Train'], color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('Conteo por clase (entrenamiento)')
axes[0].set_ylabel('Número de muestras')
axes[0].tick_params(axis='x', rotation=45)

# Barras: test
axes[1].bar(df_dist['Clase'], df_dist['Test'], color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_title('Conteo por clase (test)')
axes[1].set_ylabel('Número de muestras')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print("Conclusión: las clases están balanceadas (~6k train y ~1k test por clase).")

---
## 4. Ejemplos de muestras por clase

In [ ]:
# Una imagen de ejemplo por clase (usando el primer índice de cada clase en train)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for clase in range(10):
    idx = np.where(y_train == clase)[0][0]
    axes[clase].imshow(X_train[idx], cmap='gray')
    axes[clase].set_title(CLASS_NAMES[clase], fontsize=10)
    axes[clase].axis('off')

plt.suptitle('Una muestra por clase (entrenamiento)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Preprocesado recomendado para la red convolucional

A partir de la EDA, el preprocesado necesario es:

| Aspecto | Observación | Acción |
|--------|-------------|--------|
| **Dimensiones** | Imágenes 28×28, 1 canal | Mantener (H, W, C) o (C, H, W) según framework; añadir eje de canal si hace falta. |
| **Rango de valores** | Píxeles en [0, 255] (uint8) | **Normalización**: escalar a [0, 1] (`/ 255.0`) o estandarizar (media 0, varianza 1) para estabilidad del entrenamiento. |
| **Redimensionado** | No necesario | 28×28 es adecuado para una CNN pequeña-media. |
| **Balanceo** | Clases balanceadas | No se requiere oversampling ni ponderación de clases. |

A continuación se muestra la normalización a [0, 1] aplicada a los datos (opcional para inspección).

In [ ]:
# Ejemplo de preprocesado: normalización [0, 1] y añadir dimensión de canal
X_train_norm = X_train.astype(np.float32) / 255.0
X_test_norm = X_test.astype(np.float32) / 255.0

# Para Keras (canal al final): (N, H, W) -> (N, H, W, 1)
X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn = X_test_norm[..., np.newaxis]

print("Formas para la CNN:")
print(f"  X_train_cnn: {X_train_cnn.shape}")
print(f"  X_test_cnn:  {X_test_cnn.shape}")
print(f"  Rango de píxeles: [{X_train_cnn.min():.2f}, {X_train_cnn.max():.2f}]")

---
## Resumen EDA

- **Dataset**: Fashion-MNIST (70k imágenes, 10 clases, 28×28 escala de grises).
- **Distribución**: Clases balanceadas; no requiere balanceo.
- **Dimensiones**: Tensores (N, 28, 28); para CNN añadir canal → (N, 28, 28, 1).
- **Preprocesado**: Normalización a [0, 1] (o estandarización); opcional data augmentation en tareas posteriores.